##### Load the OpenAI API Key and Groq Chat Environment

In [23]:
from dotenv import load_dotenv
from mypy.state import state

load_dotenv()
from openAI_assistant import generate_response
from langchain_groq import ChatGroq
from groq import Groq
from langchain.llms.base import LLM

class ChatGroqLLM(LLM):
    def __init__(self, groq_api_key, model_name):
        client = Groq(
            api_key=groq_api_key,  # Replace with your Groq API key
        )

    def _call(self, prompt: str, stop=None) -> str:
        """
        Call the underlying ChatGroq LLM with the given prompt and return the response.
        """
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model=model_name,
        )
        # Use the generate method with prompt directly
        response = chat_completion.choices[0].message.content

        return response

    @property
    def _llm_type(self) -> str:
        return "chat_groq"


groq_api_key = ""
# Initialize Groq Langchain chat object and conversation
groq_chat = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)

##### Create Class Agent State

In [24]:
from typing import TypedDict, Annotated, List, Sequence, Optional
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain.schema import Document
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.graph import add_messages
from typing import Literal

class ProgramEntry(TypedDict):
    code: str
    compilation_error: Optional[str]
    event_triggered: Literal["yes", "no", ""]
    timeout: Literal["yes", "no", ""]

class AgentState(TypedDict):
    messages: List[BaseMessage]
    # messages: Annotated[Sequence[BaseMessage], add_messages]
    documents: List[Document]
    summary: str
    on_topic: str
    std_or_inline: str
    rephrased_question: str
    proceed_to_generate: bool
    re_retrieve_count: int
    question: HumanMessage
    event_is_triggered: str
    output_program: List[ProgramEntry]
    feedback: str
    re_generate_count: int
    compilation_error: str

from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()

##### STATE question_rewriter

In [25]:
def question_rewriter(state: AgentState) -> StateGraph:
    print(f"Entering question_rewriter with following state: {state}")
    state["documents"] = []
    state["on_topic"] = ""
    state["std_or_inline"] = ""
    state["rephrased_question"] = ""
    state["proceed_to_generate"] = False
    state["re_retrieve_count"] = 0
    state["event_is_triggered"] = "no"
    state["output_program"] = []
    state["feedback"] = ""
    state["re_generate_count"] = 0
    state["compilation_error"] = ""
    

    # But we don't want to reset the messages these are responsible for the conversation history
    if "messages" not in state or state["messages"] is None:
        state["messages"] = []
    if state["question"] not in state["messages"]:
        state["messages"].append(state["question"])

    # if len(state["messages"]) > 1:
    #     conversation = state["messages"][:-1]
    #     print(f"question_rewriter: Conversation: {conversation}")
    #     current_question = state["question"].content
    #     print(f"question_rewriter: Current question: {current_question}")
    #     messages = [
    #         SystemMessage(content="You are a helpful assistant that rephrases the users's question to be a standalone question optimized for retrieval.")
    #
    #     ]
    #     messages.extend(conversation) # Don't know why?
    #
    #     messages.append(HumanMessage(content=current_question))
    #     print(f"Messages==========: {messages}")
    #     rephrase_prompt = ChatPromptTemplate.from_messages(messages)
    #     llm = ChatOpenAI(model="gpt-3.5-turbo")
    #     prompt = rephrase_prompt.format()
    #     response = llm.invoke(prompt)
    #     better_question = response.content.strip()
    #     print(f"question_rewriter: Rephrased question: {better_question}")
    #     state["rephrased_question"] = better_question
    # else:
    #     state["rephrased_question"] = state["question"].content
    state["rephrased_question"] = state["question"].content
    return state

##### STATE topic_classifier

In [26]:
# Topic Classifier
class GradeTopic(BaseModel):
    #score: str = Field(description="Question is about the specified topics? If 1 or 2 or 3 -> 'Yes' else 'No'")
    score: Literal["Yes", "No"] = Field(description="Does the user input match any of the specified topics?")


def topic_classifier(state: AgentState):
    print(f"Entering topic_classifier with following state: {state}")
    system_message = SystemMessage(
        content="""You are an expert in computer architecture. You will be given a user's input (question or phrase). Your job is to determine whether the input matches **any** of the following types:

- An architectural event or its description from RISC microprocessor architecture (e.g., LSU activation, memory access initiation, detection of a page fault exception, cache invalidation request)
- A signal or net name from hardware description language (HDL) that implies an architectural event
- A snippet of HDL code

If it matches **any** of the above categories, respond only with 'Yes'. If it does not match **any**, respond only with 'No'. Do not output anything else."""
    )

    human_message = HumanMessage(
        # content=f"User's question: {state['question'].content}"
        content = f"User's question: {state['rephrased_question']}"
    )

    grade_prompt = ChatPromptTemplate.from_messages([system_message, human_message])
    # llm = ChatOpenAI(model="gpt-3.5-turbo")
    llm = groq_chat
    structured_llm = llm.with_structured_output(GradeTopic)
    grader_llm = grade_prompt | structured_llm
    result = grader_llm.invoke({})
    print(f"topic_classifier: Result: {result}")
    state["on_topic"] = result.score.strip()
    print(f"topic_classifier: On topic: {state['on_topic']}")
    return state

##### STATE event_classifier

In [27]:
# Event Classifier
class GradeEvent(BaseModel):
    score: Literal["Standard", "Inline"] = Field(
        description="Does the event require inline assembly to be triggered?"
    )

def event_classifier(state: AgentState):
    print(f"Entering event_classifier with following state: {state}")
    system_message = SystemMessage(
    content="""You are an expert in computer architecture and low-level embedded programming. You will be given a 
    user's input describing an architectural event related to OpenRISC 1000 processor.

Your task is to classify whether the event **can be triggered using only standard C code**, or if it **requires 
inline assembly inside C code**.

Use these rules:
- If the event can be triggered using common C constructs such as loads, stores, arithmetic, conditionals, or function calls, then respond with **'Standard'**
- If the event requires special-purpose registers (SPRs), privileged instructions, MMU/TLB setup, illegal opcodes, or instructions that are not guaranteed to be generated by a C compiler, then respond with **'Inline'**

**Respond with only one word: 'Standard' or 'Inline'. Do not explain.**
"""
)
    human_message = HumanMessage(
        # content=f"User's question: {state['question'].content}"
        content=f"User's question: {state['rephrased_question']}"
    )

    grade_prompt = ChatPromptTemplate.from_messages([system_message, human_message])
    llm = ChatOpenAI(model="gpt-3.5-turbo")
    llm = groq_chat
    structured_llm = llm.with_structured_output(GradeEvent)
    grader_llm = grade_prompt | structured_llm
    # grader_llm = grade_prompt | llm
    print("Before....")
    result = grader_llm.invoke({})
    print("After")
    print(f"event_classifier: Result: {result}")
    state["std_or_inline"] = result.score.strip()
    print(f"event_classifier: Standard or Inline: {state['std_or_inline']}")
    return state

##### STATE on_topic_router

In [28]:
def on_topic_router(state: AgentState):
    print(f"Entering on_topic_router with following state: {state}")
    on_topic = state["on_topic"].lower()
    if on_topic == "yes":
        print("Routing to Event Classifier")
        return "event_classifier"
    else:
        print("Routing to off_topic response")
        return "off_topic_response"

##### STATE std_or_inline_router

In [29]:
def std_or_inline_router(state: AgentState):
    print(f"Entering std_or_inline_router with following state: {state}")
    on_topic = state["std_or_inline"].lower()
    if on_topic == "standard":
        print("Routing to Standard Retriever")
        return "std_retrieve"
        # return "off_topic_response"
    else:
        print("Routing to Inline Retriever")
        return "inline_retrieve"
        # return "off_topic_response"

##### STATE off_topic_response

In [30]:
def off_topic_response(state: AgentState):
    print("Entering off_topic_response")
    if "messages" not in state or state["messages"] is None:
        state["messages"] = []
    state["messages"].append(
        AIMessage(
            content="I'm sorry, I can only answer questions about the OpenRISC 1000 high level events"
        )
    )
    return state

##### STATE couldnot_generate

In [31]:
def couldnot_generate(state: AgentState):
    print("Entering couldnot_generate")
    if "messages" not in state or state["messages"] is None:
        state["messages"] = []
    state["messages"].append(
        AIMessage(
            content="I'm sorry, I couldn't generate a program for the given event"
        )
    )
    return state

##### STATE std_retrieve

In [32]:
def std_retrieve(state: AgentState) -> StateGraph:
    print(f"Entering std_retrieve with following state: {state}")
    # documents = retriever.vectorstore.similarity_search(state["rephrased_question"])
    event = state["rephrased_question"]
    filepath = "/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.json"
    documents = find_fuzzy_event_by_high_level(filepath, event)
    print(f"std_retrieve: Retrieved documents: {documents}")
    state["documents"] = documents
    return state

##### STATE inline_retrieve

In [33]:
def inline_retrieve(state: AgentState) -> StateGraph:
    print(f"Entering inline_retrieve with following state: {state}")
    # documents = retriever.vectorstore.similarity_search(state["rephrased_question"])
    event = state["rephrased_question"]
    filepath = "/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.json"
    documents = find_fuzzy_event_by_high_level(filepath, event)
    print(f"inline_retrieve: Retrieved documents: {documents}")
    state["documents"] = documents
    return state

##### STATE retrieval_grader

In [34]:
class GradeDocument(BaseModel):
    score: str = Field(
        description="Document is relevant to the question? If yes -> 'Yes' if not -> 'No'"
    )

def retrieval_grader(state: AgentState):
    print("Entering retrieval_grader")
    system_message = SystemMessage(
        content="""You are a grader assessing the relevance of a retrieved document to a user question. Only answer 
        with 'Yes' or 'No'.
        If the document contains information relevant to the user's question, respond with 'Yes'. Otherwise, 
        respond with 'No'."""
    )
    llm = groq_chat
    structured_llm = llm.with_structured_output(GradeDocument)
    relevant_docs = []
    # for doc in state["documents"]:
    #     human_message = HumanMessage(
    #         content=f"User question: {state['rephrased_question']}\n\nRetrieved document:\n{doc.page_content}"
    #     )
    #     grade_prompt = ChatPromptTemplate.from_messages([system_message, human_message])
    #     grader_llm = grade_prompt | structured_llm
    #     result = grader_llm.invoke({})
    #     print(f"Grading document: {doc.page_content[:30]}... Result: {result.score.strip()}")
    #     if result.score.strip().lower() == "yes":
    #         relevant_docs.append(doc)
    doc = state["documents"]
    human_message = HumanMessage(
            content=f"User question: {state['rephrased_question']}\n\nRetrieved document:\n{doc}"
        )
    grade_prompt = ChatPromptTemplate.from_messages([system_message, human_message])
    grader_llm = grade_prompt | structured_llm
    result = grader_llm.invoke({})
    print(f"Grading document: \n {doc[:30]}... \n Result: {result.score.strip()}")
    if result.score.strip().lower() == "yes":
        relevant_docs.append(doc)
        state["documents"] = relevant_docs
    
    state["proceed_to_generate"] = len(relevant_docs) > 0
    print(f"retrieval_grader: Proceed to generate: {state['proceed_to_generate']}")
    return state

##### STATE proceed_retriever

In [35]:
def proceed_retriever(state: AgentState):
    print(f"Entering proceed_retriever with following state: {state}")
    re_retrieve_count = state.get("re_retrieve_count", 0)
    if state["proceed_to_generate"]:
        print("Routing to program_generator")
        return "program_generator"
        # return "couldnot_generate"
    elif not state["proceed_to_generate"] and re_retrieve_count > 3:
        print("Maximal rephrasing attempts reached. Cannot find relevant documents...")
        return "couldnot_generate"
    else:
        print("Routing to refine event details")
        return "refine_event_details"

##### STATE refine_event_details

In [36]:
def refine_event_details(state: AgentState):
    print(f"Entering refine_event_details with following state: {state}")
    re_retrieve_count = state["re_retrieve_count"]
    event_to_refine = state["rephrased_question"]
    system_message = SystemMessage(
        content="""You are a helpful assistant that slightly refines the user's question to improve retrieval results.
    Provide a slightly adjusted version of the question."""
    )
    human_message = HumanMessage(
        content=f"Original question: {event_to_refine}\n\nProvide a slightly refined question."
    )
    refine_prompt = ChatPromptTemplate.from_messages([system_message, human_message])
    llm = groq_chat
    prompt = refine_prompt.format()
    response = llm.invoke(prompt)
    refined_event = response.content.strip()
    print(f"refine_question: Refined question: {refined_event}")
    state["rephrased_question"] = refined_event
    state["re_retrieve_count"] = re_retrieve_count + 1
    return state

##### STATE program_generator

In [37]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Feedback: {feedback}
History: {history}

You are an expert in computer architecture. I will provide a high-level architectural event name along 
with a description. Your task is to write a complete C test program that triggers this event in the microprocessor.
Follow these strict rules:

- Use inline assembly code inside the C code only if the event cannot be triggered using standard C code.
- Complete C code with instructions. **DO NOT add any suggesstions or hints or "Do something" type comments**.
- Do not include any printf, puts, or output statements.
- Use the named inline assembly (e.g., `l.mfspr` instead of `.word` from Architectural manual. If the instruction requirement is 
an illegal instruction then use only `.word`).
- Do not assume a value is invalid or will cause an exception just because it looks random or unusual (e.g., 0xDEADBEEF). Instead, reason about the opcode, operand fields, and execution context.

Architectural Event: {question}
Description: {context}
Please only return the complete C code that triggers the event. Do not include any additional information or comments.
"""

# - Use inline assembly code inside the C code only if the event cannot be triggered using standard C code.
# - If a standard C construct is insufficient to trigger the event, only then use OpenRISC 1000-specific inline assembly or binary instruction encodings.
# - Use the named inline assembly (e.g., `l.mfspr`) instead of `.word` from Architectural manual if the instruction is not illegal
# - If there is error in the Feedback section below and related to unrecognized instruction, Please check the OpenRISC 1000 manual and ensure the instruction is valid and spelled exactly as defined in the ISA
# - Complete C code with instructions. No suggesstions or hints or Do something type comments.
# - Do not include any printf, puts, or output statements.
# - When inserting instructions manually (e.g., using .word or inline assembly), you must analyze the instruction encoding and ensure that it causes the specific architectural event as defined by the ISA.
# - Do not assume a value is invalid or will cause an exception just because it looks random or unusual (e.g., 0xDEADBEEF). Instead, reason about the opcode, operand fields, and execution context.
# - Avoid undefined behavior, unintentional traps, or partial instruction decoding unless it is the documented and reliable method to provoke the desired architectural response.
# - Your test programs must be deterministic and reproducible — any exception or event should be intentionally caused using well-understood architecture mechanisms.

prompt = ChatPromptTemplate.from_template(template)

rag_chain = prompt | groq_chat
# rag_chain = prompt | ChatOpenAI(model="gpt-3.5-turbo")
def format_program_history(programs: List[dict]) -> str:
    history_text = ""
    for i, p in enumerate(programs):
        history_text += f"Attempt {i + 1}:\n"
        history_text += "Program:\n"
        history_text += p["code"].strip() + "\n"
        if p["compilation_error"]:
            history_text += "Compilation Error:\n"
            history_text += p["compilation_error"].strip() + "\n"
        else:
            history_text += "Compilation: Success\n"
        history_text += f"Event Triggered: {p.get('event_triggered', '')}\n"
        history_text += "-" * 40 + "\n"
    return history_text.strip()

def program_generator(state: AgentState):
    print("Entering program_generator")
    # history = state["messages"]
    documents = state["documents"]
    rephrased_question = state["rephrased_question"]
    if state["event_is_triggered"].lower() == "yes":
        return state
    else:
        feedback = state["feedback"]
        history = format_program_history(state["output_program"])
        state["re_generate_count"] += 1
    print(f"program_generator: Feedback: {feedback}")
    print(f"program_generator: History: {history}")
    if state["std_or_inline"].lower() == "inline":
        user_query = template.format(
            feedback=feedback,
            history=history,
            context=documents,
            question=rephrased_question
        )
        response = (generate_response(assistant_inline, user_query, "11", "test1"))
        print(f"program_generator: Generated program: {response}")
        state["output_program"].append({
            "code": response.strip(),
            "compilation_error": "",
            "event_triggered": "",
            "timeout": ""
        })
    elif state["std_or_inline"].lower() == "standard":
        response = rag_chain.invoke(
            {
                "history": history,
                "feedback": feedback,
                "context": documents,
                "question": rephrased_question
            }
        )
        print(f"program_generator: Generated program: {response.content}")
        state["output_program"].append({
            "code": response.content.strip(),
            "compilation_error": "",
            "event_triggered": "",
            "timeout": ""
        })

    return state

##### STATE event_triggered

In [38]:
def find_net_for_event(file_path, target_event):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    result = None
    for i in range(1, len(lines)):
        line = lines[i].strip()
        if line.startswith("- High-Level Event:") and target_event in line:
            prev_line = lines[i - 1].strip()
            # print(f"Previous line: {prev_line}")
            if prev_line.startswith("- Net:"):
                result = prev_line.replace("- Net: ", "").strip()
                break
    return result

def event_triggered(state: AgentState):
    print("Entering event_triggered")
    latest_program = state["output_program"][-1]
    program_code = latest_program["code"]

    # Remove the Markdown code block markers if they exist
    if program_code.startswith("```c") and program_code.endswith("```"):
        program_code = program_code.strip("```c").strip("```").strip()
    # create a file name from the question
    program_file = state["rephrased_question"].replace(" ", "_").replace(":", "").replace("(", "").replace(")", "")
    # Save to a file
    arch_event_dir = "/home/m588h354/projects/autophasew/openrisc/src/architectural_events/decode_events"
    with open(
            f"{arch_event_dir}/{program_file}.c",
            "w") as file:
        file.write(program_code)
    import os
    import subprocess
    project_root = "/home/m588h354/projects/autophasew/openrisc/src"
    c_file = f"{arch_event_dir}/{program_file}.c"
    output_file = f"{arch_event_dir}/{program_file}.o"
    vcd_file_name = "decode3.vcd"
    vcd_file = "vcd_files/decode3.vcd"
    vcd_txt = "vcd_texts/decode3.txt"
    vcd_text_org = "vcd_texts/decode4.txt"
    event_details_file = "/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.txt"
    net_name  = find_net_for_event(event_details_file, state["rephrased_question"])
    print(f"Net name for event '{state['rephrased_question']}': {net_name}")
    gtkw_file = "gtkw_files/decode.gtkw"

    # Step 1: Compile with or1k-elf-gcc
    try:
        subprocess.run(
            ["/home/m588h354/projects/autophasew/openrisc/or1k-elf/bin/or1k-elf-gcc", "-O0" , c_file, "-o", output_file],
            check=True,
            capture_output=True,
            text=True
        )
        print("Compilation successful.")
    except subprocess.CalledProcessError as e:
        print("Compilation failed.")
        compilation_error = e.stderr
        print("#######################################")
        print(compilation_error)
        state["compilation_error"] = compilation_error
        print("-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-")
        print(state["compilation_error"])
        print("-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-")
        state["event_is_triggered"] = "no"
        # Update the programs metadata
        latest_program["compilation_error"] = compilation_error
        latest_program["event_triggered"] = "no"
        return state
        # exit(1)

    # Step 2: Run simulation with fusesoc
    sim_successful = True
    try:
        vcd_path = os.path.join(project_root, "testbench.vcd")
        if os.path.exists(vcd_path):
            os.remove(vcd_path)
        subprocess.run([
            "fusesoc", "run", "--target", "mor1kx_tb", "--tool", "verilator",
            "mor1kx-generic", "--elf_load", output_file,
            "--vcd", "testbench.vcd",
        ], check=True, cwd=project_root, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=3)
        latest_program["timeout"] = "no"
        print("Simulation completed.")
    except subprocess.TimeoutExpired:
        print("Simulation timed out — possible infinite loop or invalid instruction.")
        state["event_is_triggered"] = "no"
        state["compilation_error"] = "timeout"
        # Update the programs metadata
        latest_program["timeout"] = "yes"
        sim_successful = False
    except subprocess.CalledProcessError as e:
        print("Simulation failed.")
        sim_successful = False
        state["event_is_triggered"] = "no"
        print(e)
        return state

    # Step 3: Launch GTKWave using export script
    if sim_successful:
        try:
            subprocess.run([
                "./export_vcd_gtkwave.sh", gtkw_file, vcd_file_name
            ], check=True, cwd=project_root)
            print("GTKWave launched with specified VCD and GTKWave file.")
        except subprocess.CalledProcessError as e:
            print("GTKWave launch failed.")
            state["event_is_triggered"] = "no"
            print(e)
    else:
        print("Skipping GTKWave launch due to failed or timed-out simulation.")
        return state

    import os
    # Step 4: vcdcat
    try:
        with open(os.path.join(project_root, vcd_txt), "w") as txt_file:
            subprocess.run(["vcdcat", "-d", vcd_file], cwd=project_root, stdout=txt_file, check=True)
        print("VCD dumped to text.")
    except subprocess.CalledProcessError as e:
        print("vcdcat failed.")
        state["event_is_triggered"] = "no"
        print(e)
        return state

    # Step 5: grep + awk + wc -l to count non-zero activity
    try:
        result_post = subprocess.run(
            f"grep -E '{net_name}' {vcd_txt} | awk '$2 != 0' | wc -l",
            shell=True,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        result_post_cycle = subprocess.run(
            f"grep -E '{net_name}' {vcd_txt} | awk '$2 == 1 {{start=$1}} $2 == 0 {{total += ($1 - start + 1)}} END {{print total}}'",
            shell=True,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        result_pre = subprocess.run(
            f"grep -E '{net_name}' {vcd_text_org} | awk '$2 != 0' | wc -l",
            shell=True,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        result_pre_cycle = subprocess.run(
            f"grep -E '{net_name}' {vcd_text_org} | awk '$2 == 1 {{start=$1}} $2 == 0 {{total += ($1 - start + 1)}} END {{print total}}'",
            shell=True,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        print(f"Number of non-zero transitions(post) for '{net_name}': {result_post.stdout.strip()}")
        print(f"Number of non-zero transitions(pre)  for '{net_name}': {result_pre.stdout.strip()}")
        print(f"Number of cycles it was on (post) for '{net_name}': {result_post_cycle.stdout.strip()}")
        print(f"Number of cycles it was off (pre)  for '{net_name}': {result_pre_cycle.stdout.strip()}")
        count_post = int(result_post.stdout.strip())
        count_pre = int(result_pre.stdout.strip())
        cycle_post = int(result_post_cycle.stdout.strip())
        cycle_pre = int(result_pre_cycle.stdout.strip())
        
        if count_post - count_pre > 0:
            print("Event triggered successfully!")
            state["event_is_triggered"] = "yes"
            latest_program["event_triggered"] = "yes"
        elif count_post - count_pre == 0:
            if cycle_post - cycle_pre > 0:
                print("Event triggered successfully!")
                state["event_is_triggered"] = "yes"
                latest_program["event_triggered"] = "yes"
        else:
            print("Event did not trigger.")
            state["event_is_triggered"] = "no"
            latest_program["event_triggered"] = "no"
    except subprocess.CalledProcessError as e:
        print("grep/awk/wc command failed.")
        print(e)

    # state["event_is_triggered"] = "yes"
    return state

##### STATE trigger_router

In [39]:
def trigger_router(state: AgentState):
    print(f"Entering trigger_route")
    event_trigger = state["event_is_triggered"]
    print(f"Event Trigger: {event_trigger}")
    print(f"Re-generate count: {state['re_generate_count']}")
    if event_trigger == "yes":
        print("Event Triggered :D")
        return "event_trigger_success"
    elif event_trigger == "no" and state["re_generate_count"] > 2:
        print("Maximal re generation attempts reached. Cannot generate a program...")
        return "couldnot_generate"
    else:
        print("Routing to Add Compiled Log")
        return "add_compiled_log"

##### STATE add_compiled_log

In [40]:
def add_compiled_log(state: AgentState):
    print("Entering add_compiled_log")
    if state["compilation_error"].strip() == "":
        state["feedback"] = (
            f"The previous generated program did not trigger the event. Please check the History of programs where the previous programs are stored. Please reconsider the event and anlyse it further why it didn't trigger last time. Based on that change the program to trigger this event."
        )
    elif state["compilation_error"].strip() == "timeout":
        state["feedback"] = (
            f"The previous generated program compiled correctly but stucked during the simulation in the Fusesoc. Please check the History of programs where the previous programs are stored and fix it so that it can finish the simulation. Please reconsider the event and anlyse it further why it didn't trigger last time. Based on that change the program to trigger this event."
        )
    else:
        state["feedback"] = (
            f"The previous generated programs couldn't compile correctly using or1k-elf-gcc compiler. "
            # f"Here is the compilation error:\n\n{state['compilation_error']}\n\n"
            "Please check the History of programs with compilation error and based on the error, correct the program so that it compiles correctly and still triggers the event."
        )
    return state

##### PYTHON JSON-Based Retrieval 

In [41]:
import json
import difflib

def find_fuzzy_event_by_high_level(filepath, query, cutoff=0.6):
    with open(filepath, 'r') as f:
        events = json.load(f)
    
    # Build a map of HLE -> event
    hle_map = {event.get("High-Level Event", ""): event for event in events}
    hle_list = list(hle_map.keys())

    # Get best match
    matches = difflib.get_close_matches(query, hle_list, n=1, cutoff=cutoff)
    
    if matches:
        best_match = matches[0]
        event = hle_map[best_match]
        return (
            f"Net: {event.get('Net')}\n"
            f"High-Level Event: {event.get('High-Level Event')}\n"
            f"Logical Summary: {event.get('Logical Summary')}\n"
            f"Reasoning: {event.get('Reasoning')}"
        )
    return "Event not found."

# Example usage
# filepath = "/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.json"
# query = "Identification of load and store instruction storage operations"
# result = find_fuzzy_event_by_high_level(filepath, query)
# print(result)


##### OpenAI File Assistant

In [42]:
from openai import OpenAI
client = OpenAI(
    api_key = "",
    organization= None
)
decode_events_id = "asst_yBrlPq2dAcuSa1er6Ghxcfep"
openRISC_manual_id = "asst_UoYBmjJNgNNznOfJRoZMGPDx"
openrisc_and_decode = "asst_uVQRYMJrt4bE96HDDvqEg954"
# assistant_inline = client.beta.assistants.retrieve(openRISC_manual_id)
assistant = client.beta.assistants.retrieve(decode_events_id)
assistant_inline = client.beta.assistants.retrieve(openrisc_and_decode)
# response = generate_response(assistant_inline, "ISA for event set flag if equal","3", "test2")
# response
# --------------------------------------------------------------
# Update assistant
# --------------------------------------------------------------
assistant_inline = client.beta.assistants.update(
    assistant_id=assistant_inline.id,
    # model="o3-mini",
    model="gpt-4o-mini-2024-07-18",
    # model="gpt-3.5-turbo-1106",
    # tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

##### GRAPH Creation

In [43]:
workflow = StateGraph(AgentState)
workflow.add_node("question_rewriter", question_rewriter)
workflow.add_node("topic_classifier", topic_classifier)
workflow.add_node("std_retrieve", std_retrieve)
workflow.add_node("inline_retrieve", inline_retrieve)
workflow.add_node("event_classifier", event_classifier)
workflow.add_node("retrieval_grader", retrieval_grader)
workflow.add_node("program_generator", program_generator)
workflow.add_node("event_triggered", event_triggered)
workflow.add_node("add_compiled_log", add_compiled_log)
workflow.add_node("refine_event_details", refine_event_details)
workflow.add_node("off_topic_response", off_topic_response)
workflow.add_node("couldnot_generate", couldnot_generate)

workflow.add_edge("question_rewriter", "topic_classifier")
workflow.add_conditional_edges(
    "topic_classifier",
    on_topic_router,
    {
        "off_topic_response": "off_topic_response",
        "event_classifier": "event_classifier"
    }
)
workflow.add_conditional_edges(
    "event_classifier",
    std_or_inline_router,
    {
        "std_retrieve": "std_retrieve",
        "inline_retrieve": "inline_retrieve"
        # "off_topic_response": "off_topic_response",
        # "off_topic_response": "off_topic_response"
    }
)
workflow.add_edge("std_retrieve", "retrieval_grader")

workflow.add_conditional_edges(
    "retrieval_grader",
    proceed_retriever,
    {
        "couldnot_generate": "couldnot_generate",
        "refine_event_details": "refine_event_details",
        "program_generator": "program_generator",

    }
)
workflow.add_edge("inline_retrieve", "program_generator")
workflow.add_edge("refine_event_details", "std_retrieve")
workflow.add_edge("program_generator", "event_triggered")
workflow.add_conditional_edges(
    "event_triggered",
    trigger_router,
    {
        "couldnot_generate": "couldnot_generate",
        "event_trigger_success": END,
        "add_compiled_log": "add_compiled_log",
    }
)
workflow.add_edge("add_compiled_log", "program_generator")
workflow.add_edge("off_topic_response", END)
workflow.add_edge("couldnot_generate", END)

workflow.set_entry_point("question_rewriter")
graph = workflow.compile(checkpointer=checkpointer)

# from langchain_core.runnables.graph import MermaidDrawMethod
# from PIL import Image
# img_bytes = graph.get_graph().draw_mermaid_png(
#     draw_method=MermaidDrawMethod.API,
# )

# # Save to a file
# with open("workflow_graph.png", "wb") as f:
#     f.write(img_bytes)

# # Optional: Display it using PIL (pillow)
# img = Image.open("workflow_graph.png")
# img.show()  # This will open the image using your OS default viewer

#### Framework testing .....

In [44]:
input_data = {"question": HumanMessage(content="Register File Destination Address Generation")}
response = graph.invoke(input_data, config={"configurable": {"thread_id": 22}})

Entering question_rewriter with following state: {'question': HumanMessage(content='Instruction Bus Alignment Exception Detection', additional_kwargs={}, response_metadata={})}
Entering topic_classifier with following state: {'messages': [HumanMessage(content='Instruction Bus Alignment Exception Detection', additional_kwargs={}, response_metadata={})], 'documents': [], 'on_topic': '', 'std_or_inline': '', 'rephrased_question': 'Instruction Bus Alignment Exception Detection', 'proceed_to_generate': False, 're_retrieve_count': 0, 'question': HumanMessage(content='Instruction Bus Alignment Exception Detection', additional_kwargs={}, response_metadata={}), 'event_is_triggered': 'no', 'output_program': [], 'feedback': '', 're_generate_count': 0, 'compilation_error': ''}
topic_classifier: Result: score='Yes'
topic_classifier: On topic: Yes
Entering on_topic_router with following state: {'messages': [HumanMessage(content='Instruction Bus Alignment Exception Detection', additional_kwargs={}, r


GTKWave Analyzer v3.3.104 (w)1999-2020 BSI

[0] start time.
[19566] end time.
GTKWAVE | Saving VCD...
GTKWAVE | VCD written successfully.


Exiting.
GTKWave launched with specified VCD and GTKWave file.
VCD dumped to text.
Number of non-zero transitions(post) for 'decode_except_ibus_align': 0
Number of non-zero transitions(pre)  for 'decode_except_ibus_align': 0
Number of cycles it was on (post) for 'decode_except_ibus_align': 1
Number of cycles it was off (pre)  for 'decode_except_ibus_align': 1
Entering trigger_route
Event Trigger: no
Re-generate count: 1
Routing to Add Compiled Log
Entering add_compiled_log
Entering program_generator
program_generator: Feedback: The previous generated program did not trigger the event. Please check the History of programs where the previous programs are stored. Please reconsider the event and anlyse it further why it didn't trigger last time. Based on that change the program to trigger this event.
program_generator: History: [{'code': '```c\nint main() {\n    asm volatile (\n        ".word 0x00000008"  // Instruction that will trigger an instruction bus alignment exception\n    );\n    


GTKWave Analyzer v3.3.104 (w)1999-2020 BSI

[0] start time.
[19564] end time.
GTKWAVE | Saving VCD...
GTKWAVE | VCD written successfully.


Exiting.
GTKWave launched with specified VCD and GTKWave file.
VCD dumped to text.
Number of non-zero transitions(post) for 'decode_except_ibus_align': 0
Number of non-zero transitions(pre)  for 'decode_except_ibus_align': 0
Number of cycles it was on (post) for 'decode_except_ibus_align': 1
Number of cycles it was off (pre)  for 'decode_except_ibus_align': 1
Entering trigger_route
Event Trigger: no
Re-generate count: 2
Routing to Add Compiled Log
Entering add_compiled_log
Entering program_generator
program_generator: Feedback: The previous generated program did not trigger the event. Please check the History of programs where the previous programs are stored. Please reconsider the event and anlyse it further why it didn't trigger last time. Based on that change the program to trigger this event.
program_generator: History: [{'code': '```c\nint main() {\n    asm volatile (\n        ".word 0x00000008"  // Instruction that will trigger an instruction bus alignment exception\n    );\n    


GTKWave Analyzer v3.3.104 (w)1999-2020 BSI

[0] start time.
[19562] end time.


Exiting.
GTKWave launched with specified VCD and GTKWave file.
VCD dumped to text.
Number of non-zero transitions(post) for 'decode_except_ibus_align': 0
Number of non-zero transitions(pre)  for 'decode_except_ibus_align': 0
Number of cycles it was on (post) for 'decode_except_ibus_align': 1
Number of cycles it was off (pre)  for 'decode_except_ibus_align': 1
Entering trigger_route
Event Trigger: no
Re-generate count: 3
Maximal re generation attempts reached. Cannot generate a program...
Entering couldnot_generate


##### ETC

In [ ]:
import json
from langchain_core.messages import HumanMessage
json_file = "/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.json"
# Load the JSON file
with open(json_file, "r") as f:
    event_data = json.load(f)

# Slice the list for indices 12 to 20 (inclusive)
# Python uses 0-based indexing, so this is actually elements 12 to 20 → [12:21]
event_subset = event_data[18:28]
# Dictionary to store results
results = {}

# Loop through each event entry
for entry in event_subset:
    question = entry.get("High-Level Event", "").strip()
    if not question:
        continue  # Skip if there's no valid event
    print(question)
    # input_data = {"question": HumanMessage(content=question)}
    # response = graph.invoke(input_data, config={"configurable": {"thread_id": 21}})
    # results[question] = response.get("event_is_triggered")

# # Print the final results
# for event_name, triggered in results.items():
#     print(f"Event: {event_name} => Triggered: {triggered}")


##### Trigger test

In [ ]:
def find_net_for_event(file_path, target_event):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    result = None
    for i in range(1, len(lines)):
        line = lines[i].strip()
        if line.startswith("- High-Level Event:") and target_event in line:
            prev_line = lines[i - 1].strip()
            # print(f"Previous line: {prev_line}")
            if prev_line.startswith("- Net:"):
                result = prev_line.replace("- Net: ", "").strip()
                break
    return result

def event_triggered():
    print("Entering event_triggered")
    program_code = """
#include <stdint.h>

int main(void) {
    asm volatile (
        ".word 0xc28002c5"   // Encoding for the Move from Special Purpose Register (mfspr r5, r6, 709) instruction
    );
    return 0;
}
    """
    event_name = "Move from Special Purpose Register instruction decoding"
    # Remove the Markdown code block markers if they exist
    if program_code.startswith("```c") and program_code.endswith("```"):
        program_code = program_code.strip("```c").strip("```").strip()
    # create a file name from the question
    program_file = event_name.replace(" ", "_").replace(":", "").replace("(", "").replace(")", "")
    # Save to a file
    arch_event_dir = "/home/m588h354/projects/autophasew/openrisc/src/architectural_events/decode_events"
    with open(
            f"{arch_event_dir}/{program_file}.c",
            "w") as file:
        file.write(program_code)

    import subprocess
    project_root = "/home/m588h354/projects/autophasew/openrisc/src"
    c_file = f"{arch_event_dir}/{program_file}.c"
    output_file = f"{arch_event_dir}/{program_file}.o"
    vcd_file_name = "decode3.vcd"
    vcd_file = "vcd_files/decode3.vcd"
    vcd_txt = "vcd_texts/decode3.txt"
    vcd_text_org = "vcd_texts/decode4.txt"
    event_details_file = "/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.txt"
    net_name  = find_net_for_event(event_details_file, event_name)
    print(f"Net name for event '{event_name}': {net_name}")
    gtkw_file = "gtkw_files/decode.gtkw"

    # Step 1: Compile with or1k-elf-gcc
    try:
        subprocess.run(
            ["/home/m588h354/projects/autophasew/openrisc/or1k-elf/bin/or1k-elf-gcc", c_file, "-o", output_file],
            check=True,
            capture_output=True,
            text=True
        )
        print("Compilation successful.")
    except subprocess.CalledProcessError as e:
        print("Compilation failed.")
        compilation_error = e.stderr
        print("C____________E:", compilation_error)
        # print(e)
        return

    # Step 2: Run simulation with fusesoc
    sim_successful = True
    try:
        subprocess.run([
            "fusesoc", "run", "--target", "mor1kx_tb", "--tool", "verilator",
            "mor1kx-generic", "--elf_load", output_file,
            "--vcd", "testbench.vcd",
        ], check=True, cwd=project_root, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=3)
        print("Simulation completed.")
    except subprocess.TimeoutExpired:
        print("Simulation timed out — possible infinite loop or invalid instruction.")
        # state["event_is_triggered"] = "no"
        sim_successful = False
    except subprocess.CalledProcessError as e:
        print("Simulation failed.")
        print(e)
        sim_successful = False

    # Step 3: Launch GTKWave using export script
    if sim_successful:
        try:
            subprocess.run([
                "./export_vcd_gtkwave.sh", gtkw_file, vcd_file_name
            ], check=True, cwd=project_root)
            print("GTKWave launched with specified VCD and GTKWave file.")
        except subprocess.CalledProcessError as e:
            print("GTKWave launch failed.")
            print(e)
    else :
        print("Skipping GTKWave launch due to failed or timed-out simulation.")
        return


    import os
    # Step 4: vcdcat
    try:
        with open(os.path.join(project_root, vcd_txt), "w") as txt_file:
            subprocess.run(["vcdcat", "-d", vcd_file], cwd=project_root, stdout=txt_file, check=True)
        print("VCD dumped to text.")
    except subprocess.CalledProcessError as e:
        print("vcdcat failed.")
        print(e)

    # Step 5: grep + awk + wc -l to count non-zero activity
    try:
        result_post = subprocess.run(
            f"grep -E '{net_name}' {vcd_txt} | awk '$2 != 0' | wc -l",
            shell=True,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        result_pre = subprocess.run(
            f"grep -E '{net_name}' {vcd_text_org} | awk '$2 != 0' | wc -l",
            shell=True,
            cwd=project_root,
            capture_output=True,
            text=True,
            check=True
        )
        print(f"Number of non-zero transitions(post) for '{net_name}': {result_post.stdout.strip()}")
        print(f"Number of non-zero transitions(pre)  for '{net_name}': {result_pre.stdout.strip()}")
        count_post = int(result_post.stdout.strip())
        count_pre = int(result_pre.stdout.strip())
        if count_post - count_pre > 0:
            print("Event triggered successfully!")
        else:
            print("Event did not trigger.")
    except subprocess.CalledProcessError as e:
        print("grep/awk/wc command failed.")
        print(e)

event_triggered()

##### Custom Retriever

In [ ]:
from langchain.schema import Document
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader

embedding_fn = OpenAIEmbeddings()

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
pages = []
manuals = [
    # PyPDFLoader("/home/m588h354/projects/Rare_net_analysis-repo/event_identification/openrisc-arch-1.4-rev0.pdf"),
    # TextLoader("/home/m588h354/projects/Rare_net_analysis-repo/event_identification/or1k-sprs.h"),
    # TextLoader("/home/m588h354/projects/Rare_net_analysis-repo/event_identification/or1k-support.h"),
    TextLoader("/home/m588h354/projects/Rare_net_analysis-repo/event_identification/event_files/HIGH_LEVEL_EVENTS_DECODE.txt"),
    ]
for manual in manuals:
    pages.extend(manual.load())

splitter = RecursiveCharacterTextSplitter(
    # chunk_size=800, chunk_overlap=100, separators=["\n\n", "\n", ".", " "]
    chunk_size=10000
)

docs = splitter.split_documents(pages)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="full_documents", embedding_function=OpenAIEmbeddings()
)
# db = Chroma.from_documents(docs, embedding_fn)
# retriever = db.as_retriever(search_kwargs={"k": 2})
import uuid

from langchain.retrievers.multi_vector import MultiVectorRetriever
# The storage layer for the parent documents
store = InMemoryByteStore()
id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

doc_ids = [str(uuid.uuid4()) for _ in docs]
child_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100, separators=["\n\n", "\n", ".", " "]
    #chunk_size=400
)
sub_docs = []
for i, doc in enumerate(docs):
    _id = doc_ids[i]
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    sub_docs.extend(_sub_docs)


retriever.vectorstore.add_documents(sub_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

from langchain.retrievers.multi_vector import SearchType
retriever.search_type = SearchType.mmr

def get_retriever_prompt(event_description: str) -> str:
    return f"""
You're preparing to generate a test program for an architectural event: "{event_description}".

Your primary goal is to use minimal inline assembly to trigger this event.
Retrieve only the information from architecture manuals that is relevant to use inline assembly to trigger this
event. Your response will pass to the test program generator to create a test program that intentionally triggers this event.
"""
# - What instructions from ISA is relevant for inline assembly?
# - What causes this event?
# - Which SPRs or privilege levels are relevant (if any)?
# - Does this event require special setup (e.g., MMU enabled, exceptions enabled)?
#
# Do not include general background unless it directly relates to triggering this event.
# """
query = get_retriever_prompt("signal pipeline_flush_i")
input = "What causes this event : set flag if equal?"
#The vector store alone will retrieve small chunks:
response = retriever.vectorstore.similarity_search(input, k = 3)
# for i in response:
#     print(i)
#     print("============================================")

#Whereas the retriever will return the larger parent document:
#len(retriever.invoke("justice breyer")[0].page_content)
